# Research models for turbofan RUL

The temporal CNN uses dilated convolutions and the GRU uses gated recurrent memory over the same 30 observed cycles. Both add the latest sensor reading to the regression head. The SNN uses 64 LIF neurons and receives either ground truth alone, CNN targets, GRU targets, or an equal average of the two frozen teachers. Each guided student uses the same 60% ground-truth and 40% teacher objective. The supervised student controls for capacity and the training target.

Training uses raw RUL divided by 125 for numerical scale. The original experiments capped training labels, so this comparison changes the target as well as the architecture. Checkpoint selection uses raw validation RMSE on held-out training engines. Test files are opened after all selections have been saved. This first experiment uses seed 42 across four subsets.

[Convolutional RUL prediction](https://escholarship.org/uc/item/5ns8r3fs) · [Gated recurrent models](https://aclanthology.org/D14-1179/) · [Knowledge distillation](https://arxiv.org/abs/1503.02531)

## Completed run

These saved tables come from the local Ubuntu CPU experiment. The training cells below reproduce the experiment and save new results to Drive. Negative RMSE differences favor the student. The single seed limits conclusions about repeatability.


In [1]:
from io import StringIO
import pandas as pd
from IPython.display import display

print('local Ubuntu CPU', '· seed 42 · official raw test set')
metrics = pd.read_csv(StringIO('subset,seed,model,RMSE,MAE,MSE,RAE_pct,R2,cmapss_score,bias_cycles\r\nFD001,42,temporal_cnn,20.904978157678766,15.063967342935502,437.01811177302625,40.9712116857838,0.7469304504676981,2275.0097418789,2.5451420215144753\r\nFD001,42,gru_memory,22.97791400876285,16.380864368453622,527.9845321941,44.55292861151684,0.6942533864779945,9743.494713199609,8.162604366913438\r\nFD001,42,snn_supervised,23.631367682807625,18.19109906628728,558.4415385600446,49.476432979088095,0.6766162664742927,5897.882451335602,10.361664343178273\r\nFD001,42,snn_from_cnn,22.85348320307469,17.49768218331039,522.281694513217,47.590467001322885,0.6975558000944501,2456.4010514640145,6.606933103427291\r\nFD001,42,snn_from_gru,25.592305979970508,19.133136817738414,654.9661253724341,52.03860184549929,0.6207205654830947,5025.009716403588,10.070100893750787\r\nFD001,42,snn_from_both,22.171059973013104,16.51036339186132,491.5559003269438,44.905142061025366,0.715348570426543,2735.440682254253,7.451109701320529\r\nFD001,42,teacher_average,21.151133304768905,14.828237852472812,447.37044007610433,40.330070966711666,0.7409355981041623,4083.667065643761,5.353873194213957\r\nFD002,42,temporal_cnn,26.317966775033405,17.887775094611534,692.6353751717622,39.500065874215096,0.7605126667855359,9269.060094925762,-3.0141557831043904\r\nFD002,42,gru_memory,25.599237012563865,17.703053528896056,655.3209356254197,39.09216081192775,0.7734146003824589,9125.970492556724,-3.44108325126491\r\nFD002,42,snn_supervised,28.35033752936066,20.13673841835334,803.7416380286755,44.46626200350651,0.7220962884877498,29334.86651795978,-0.566489758092234\r\nFD002,42,snn_from_cnn,26.836055880386546,18.461770456327436,720.1738952152293,40.76757144600093,0.7509908794753692,13769.2765262665,-2.7872391276953303\r\nFD002,42,snn_from_gru,27.005680216174103,19.061099912784275,729.3067639382573,42.09102016364128,0.7478330760285941,12341.689827327771,-4.300600648616732\r\nFD002,42,snn_from_both,27.912237673810065,19.340865380659295,779.0930119592618,42.70880266325035,0.7306188588564673,12104.101145532275,-5.730467119415985\r\nFD002,42,teacher_average,25.330500620082013,17.445538001494576,641.6342616639752,38.52351098028626,0.7781469388755504,8118.096614568915,-3.22761951718465\r\nFD003,42,temporal_cnn,25.48840734694817,16.904700419679283,649.6589090839614,47.22827661838788,0.6208801243162564,236129.9019035635,9.06967983238399\r\nFD003,42,gru_memory,27.016533571635566,18.057582507133485,729.8930862273115,50.44919345115742,0.5740580599393279,1459678.8162863369,9.272388533055782\r\nFD003,42,snn_supervised,44.16554832622508,30.501260478124024,1950.5956589561229,85.21428545361188,-0.1383043831037829,6153020.033187052,20.229572417512536\r\nFD003,42,snn_from_cnn,34.240696405623176,24.62777810033411,1172.4252903420556,68.8049765889268,0.3158106136807991,254999.74918310693,18.083280251659453\r\nFD003,42,snn_from_gru,32.64629199641072,22.197368330545725,1065.7803811149106,62.01490861647256,0.3780451249961423,5343003.752425299,15.351431268565356\r\nFD003,42,snn_from_both,31.579569629256536,21.974346007183193,997.2692179690619,61.39182984439451,0.4180260184952046,248189.80744581422,13.945425344929099\r\nFD003,42,teacher_average,25.638505144102744,16.961260864324867,657.3329460241829,47.38629493631506,0.6164018051704887,584985.383685091,9.171034182719886\r\nFD004,42,temporal_cnn,29.186913209012886,20.997215180597717,851.8759026704508,45.183549713572646,0.7134474950235892,38031.2163179026,0.5819544813505584\r\nFD004,42,gru_memory,29.39766028443055,19.78994961326281,864.2224301987852,42.585655501888496,0.7092943919954207,290912.1413609074,7.203017702596562\r\nFD004,42,snn_supervised,32.20760371612056,22.77431881677119,1037.329737134663,49.00766875986462,0.6510648631676923,921893.0988486105,4.80751849786048\r\nFD004,42,snn_from_cnn,31.758484589363057,22.968596282355005,1008.6013434128107,49.425731129037494,0.660728469285836,175700.12771105726,3.778705459178215\r\nFD004,42,snn_from_gru,32.362508084737556,23.618928240912577,1047.3319295347037,50.82516939392527,0.6477003434313047,289914.2646427079,6.997151194560912\r\nFD004,42,snn_from_both,29.851896262750902,22.023197886233607,891.1357104820413,47.39134442285396,0.7002413505158771,112627.61528583254,1.1447421907477322\r\nFD004,42,teacher_average,27.553377589673673,19.20265259455529,759.1886165991314,41.321861050425966,0.7446254798919749,61545.520082392286,3.8924860919735607\r\n'))
pairs = pd.read_csv(StringIO('subset,seed,split,target,teacher,student,teacher_RMSE,student_RMSE,student_minus_teacher,student_minus_supervised\r\nFD001,42,test,raw,temporal_cnn,snn_from_cnn,20.904978157678766,22.85348320307469,1.9485050453959225,-0.7778844797329363\r\nFD001,42,test,raw,gru_memory,snn_from_gru,22.97791400876285,25.592305979970508,2.6143919712076595,1.9609382971628833\r\nFD001,42,test,raw,teacher_average,snn_from_both,21.151133304768905,22.171059973013104,1.0199266682441994,-1.4603077097945203\r\nFD002,42,test,raw,temporal_cnn,snn_from_cnn,26.317966775033405,26.836055880386546,0.5180891053531411,-1.5142816489741158\r\nFD002,42,test,raw,gru_memory,snn_from_gru,25.599237012563865,27.005680216174103,1.4064432036102374,-1.3446573131865591\r\nFD002,42,test,raw,teacher_average,snn_from_both,25.330500620082013,27.912237673810065,2.5817370537280517,-0.43809985555059683\r\nFD003,42,test,raw,temporal_cnn,snn_from_cnn,25.48840734694817,34.240696405623176,8.752289058675007,-9.924851920601903\r\nFD003,42,test,raw,gru_memory,snn_from_gru,27.016533571635566,32.64629199641072,5.629758424775151,-11.519256329814361\r\nFD003,42,test,raw,teacher_average,snn_from_both,25.638505144102744,31.579569629256536,5.941064485153792,-12.585978696968542\r\nFD004,42,test,raw,temporal_cnn,snn_from_cnn,29.186913209012886,31.758484589363057,2.5715713803501714,-0.4491191267575019\r\nFD004,42,test,raw,gru_memory,snn_from_gru,29.39766028443055,32.362508084737556,2.9648478003070053,0.15490436861699663\r\nFD004,42,test,raw,teacher_average,snn_from_both,27.553377589673673,29.851896262750902,2.298518673077229,-2.355707453369657\r\n'))
display(pairs.round(3))
display(metrics.round(3))


local Ubuntu CPU · seed 42 · official raw test set
   subset  seed  ...  student_minus_teacher  student_minus_supervised
0   FD001    42  ...                  1.949                    -0.778
1   FD001    42  ...                  2.614                     1.961
2   FD001    42  ...                  1.020                    -1.460
3   FD002    42  ...                  0.518                    -1.514
4   FD002    42  ...                  1.406                    -1.345
5   FD002    42  ...                  2.582                    -0.438
6   FD003    42  ...                  8.752                    -9.925
7   FD003    42  ...                  5.630                   -11.519
8   FD003    42  ...                  5.941                   -12.586
9   FD004    42  ...                  2.572                    -0.449
10  FD004    42  ...                  2.965                     0.155
11  FD004    42  ...                  2.299                    -2.356

[12 rows x 10 columns]
   subset  seed

In [ ]:
from pathlib import Path
import subprocess
import sys
from zipfile import ZipFile
import torch
from google.colab import drive

drive.mount("/content/drive")
drive_root = Path("/content/drive/MyDrive")
data_root = drive_root / "ML projec/SpikeDistill-RUL/artifacts/data"
bundle = drive_root / "SpikeDistill-RUL - Project Demo/04 - Research extensions/Research_Models_Source.zip"
repo = Path("/content/spikedistill_research_models")
with ZipFile(bundle) as source:
    source.extractall(repo)
run_root = drive_root / "ML projec/SpikeDistill-RUL/colab_runs/research_teachers_v1"
device = "cuda" if torch.cuda.is_available() else "cpu"
subsets = ["FD001", "FD002", "FD003", "FD004"]
seeds = ["42"]
print(f"Device {device}. Checkpoints and scores are saved to Drive.")

## Train and select on validation

In [ ]:
command = [sys.executable, "-u", "-m", "research_extensions.teacher_student",
           "--data-root", str(data_root), "--output", str(run_root),
           "--subsets", *subsets, "--seeds", *seeds, "--device", device]
subprocess.run([*command, "--phase", "train"], cwd=repo, check=True)

## Evaluate the frozen checkpoints on the official test set

In [ ]:
subprocess.run([*command, "--phase", "test"], cwd=repo, check=True)

## Paired teacher and student tables

The two-teacher student is compared with the equal-average teacher prediction. A negative student-minus-teacher difference means the student achieved lower RMSE. The comparison uses the same engine endpoints and raw cycle targets for every model.

In [ ]:
sys.path.insert(0, str(repo))
from research_extensions.teacher_student_report import report

report(run_root, subsets, [int(seed) for seed in seeds], repo / "results/test_metrics_per_seed.csv")